In [20]:
# Cell 1: Import Libraries (Modified)
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from datetime import datetime as dt

# --- TRY CHANGING THIS LINE ---
pio.renderers.default = "browser"
# --- END OF CHANGE ---

print("Libraries imported.")

# Re-run cells 2, 3, and 4 after changing this

Libraries imported.


In [12]:
# Cell 2: Load Data
# --- IMPORTANT: Update this path if your file is located elsewhere ---
file_path = '/Users/runi/Downloads/processed_data.csv'

try:
    df = pd.read_csv(file_path, index_col=0)
    # Convert date column
    df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')
    print(f"Data loaded successfully from {file_path}")
    print(f"DataFrame shape: {df.shape}")
    print("First 5 rows:")
    print(df.head())
    print("\nData types:")
    print(df.dtypes)
except FileNotFoundError:
    print(f"Error: Data file not found at {file_path}. Please check the path.")
    df = None # Set df to None if file not found
except Exception as e:
    print(f"An error occurred while loading the data: {e}")
    df = None

Data loaded successfully from /Users/runi/Downloads/processed_data.csv
DataFrame shape: (895485, 14)
First 5 rows:
                  rideable_type           started_at             ended_at  \
ride_id                                                                     
3255D3E3F33CDC45   classic_bike  2022-03-18 15:38:17  2022-03-18 15:45:34   
17FA5604A37338F9  electric_bike  2022-03-04 16:44:48  2022-03-04 16:50:45   
7DEC9ADDB8D6BBE1  electric_bike  2022-03-13 17:44:32  2022-03-13 17:54:44   
9D69F74EEF231A2E   classic_bike  2022-03-13 15:33:47  2022-03-13 15:41:22   
C84AE4A9D78A6347   classic_bike  2022-03-11 12:21:18  2022-03-11 12:33:24   

                                      start_station_name start_station_id  \
ride_id                                                                     
3255D3E3F33CDC45  Mama Johnson Field - 4 St & Jackson St            HB404   
17FA5604A37338F9                   Baldwin at Montgomery            JC020   
7DEC9ADDB8D6BBE1                   Ba

In [13]:
# Cell 3: Data Wrangling for Bar Chart
if df is not None:
    # Add a 'value' column for counting trips
    df['value'] = 1

    # Group by start station and sum the trips
    df_groupby_bar = df.groupby('start_station_name', as_index=False).agg({'value': 'sum'})

    # Get the top 20 stations
    top20 = df_groupby_bar.nlargest(20, 'value')
    print("\nTop 20 most popular start stations:")
    print(top20)
else:
    print("DataFrame not loaded, skipping bar chart wrangling.")


Top 20 most popular start stations:
                              start_station_name  value
37                                 Grove St PATH  42556
75  South Waterfront Walkway - Sinatra Dr & 1 St  34245
44       Hoboken Terminal - River St & Hudson Pl  33020
43      Hoboken Terminal - Hudson St & Hudson Pl  30244
22              City Hall - Washington St & 1 St  23289
69                                  Newport Pkwy  21961
38                                 Hamilton Park  20477
68                                  Newport PATH  19965
42                    Hoboken Ave at Monmouth St  19501
61                              Marin Light Rail  19230
1                           12 St & Sinatra Dr N  17137
2              14 St Ferry - 14 St & Shipyard Ln  17066
25                    Columbus Dr at Exchange Pl  16731
45                              Hudson St & 4 St  15883
39                                    Harborside  15754
0                          11 St & Washington St  15502
81         

In [30]:
# Cell 4: Create and Show Bar Chart
pio.renderers.default = "browser"

if df is not None and not top20.empty:
    fig_bar = px.bar(
        top20,
        x='start_station_name',
        y='value',
        title='Top 20 Most Popular Start Stations (Jersey City/Hoboken)',
        labels={'start_station_name': 'Start Station', 'value': 'Number of Trips'},
        color='value',  # Color bars based on the number of trips
        color_continuous_scale='Blues' # Use a blue color scale
    )

    # Customize layout
    fig_bar.update_layout(
        xaxis_tickangle=-45, # Rotate x-axis labels for better readability
        xaxis_title='Start Station',
        yaxis_title='Number of Trips',
        title_x=0.5 # Center the title
    )

    # Show the plot in the notebook
    fig_bar.show()
else:
    print("Skipping bar chart display due to previous errors or empty data.")

In [16]:
# Cell 5: Data Wrangling for Dual-Axis Line Chart
if df is not None:
    # Aggregate bike rides per day
    # Check if 'value' column exists from previous step
    if 'value' in df.columns:
        daily_rides = df.groupby('date').agg(bike_rides_daily=('value', 'sum')).reset_index()

        # Aggregate average temperature per day (assuming 'avgTemp' is daily or take the mean)
        # If 'avgTemp' already represents the daily average, we just need to select unique values per day.
        # If 'avgTemp' varies within a day, we should average it. Let's assume it's daily for now.
        daily_temp = df[['date', 'avgTemp']].drop_duplicates().sort_values('date').reset_index(drop=True)
        # If you need to average temp per day:
        # daily_temp = df.groupby('date').agg(avgTemp=('avgTemp', 'mean')).reset_index()


        # Merge the two aggregated dataframes
        df_line = pd.merge(daily_rides, daily_temp, on='date', how='left')

        print("\nData prepared for line chart:")
        print(df_line.head())
    else:
         print("Skipping line chart wrangling because 'value' column is missing.")
         df_line = None
else:
    print("DataFrame not loaded, skipping line chart wrangling.")
    df_line = None


Data prepared for line chart:
        date  bike_rides_daily  avgTemp
0 2022-01-01               592     11.6
1 2022-01-02              1248     11.4
2 2022-01-03               832      1.4
3 2022-01-04               934     -2.7
4 2022-01-05               914      3.2


In [25]:
# Cell 6: Create and Show Dual-Axis Line Chart

pio.renderers.default = "browser"

if df_line is not None and not df_line.empty:
    # Create figure with secondary y-axis
    fig_line = make_subplots(specs=[[{"secondary_y": True}]])

    # Add Bike Rides trace (Primary Y-axis - left)
    fig_line.add_trace(
        go.Scatter(x=df_line['date'], y=df_line['bike_rides_daily'], name='Daily Bike Rides'),
        secondary_y=False,
    )

    # Add Temperature trace (Secondary Y-axis - right)
    fig_line.add_trace(
        go.Scatter(x=df_line['date'], y=df_line['avgTemp'], name='Average Daily Temperature'),
        secondary_y=True,
    )

    # Add figure title
    fig_line.update_layout(
        title_text='Daily Bike Rides vs. Average Temperature',
        title_x=0.5
    )

    # Set x-axis title
    fig_line.update_xaxes(title_text='Date')

    # Set y-axes titles
    fig_line.update_yaxes(title_text='Number of Bike Rides', secondary_y=False)
    fig_line.update_yaxes(title_text='Average Temperature', secondary_y=True)

    # Show the plot in the notebook
    fig_line.show()
else:
     print("Skipping line chart display due to previous errors or empty data.")